# 1. Setup


In [38]:
import findspark, os
os.environ["SPARK_HOME"] = "/home/maxence/Documents/data_engineering/spark-4.0.1-bin-hadoop3"
findspark.init()

In [39]:
# TODO: Set the path to a1-brand.csv
DATA_PATH = "/home/maxence/Documents/data_engineering/a1-brand.csv"

In [40]:
import sys, re
from pyspark.sql import SparkSession, functions as F, types as T
from pyspark.sql.functions import col
from pyspark.sql.functions import col, lower, regexp_replace, split, explode

In [41]:
from IPython.core.magic import register_cell_magic
import time, os, platform
import psutil, resource

def _rss_bytes():
    return psutil.Process(os.getpid()).memory_info().rss

def _ru_maxrss_bytes():
    # ru_maxrss: bytes on macOS; kilobytes on Linux
    ru = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
    if platform.system() == "Darwin":
        return int(ru)  # bytes
    else:
        return int(ru) * 1024  # KB -> bytes

@register_cell_magic
def timemem(line, cell):
    """
    Measure wall time and memory around the execution of this cell.
    Usage:
        %%timemem
        <your code>
    """
    ip = get_ipython()
    rss_before = _rss_bytes()
    peak_before = _ru_maxrss_bytes()
    t0 = time.perf_counter()

    # Execute the cell body
    result = ip.run_cell(cell)

    t1 = time.perf_counter()
    rss_after = _rss_bytes()
    peak_after = _ru_maxrss_bytes()

    wall = t1 - t0
    rss_delta_mb = (rss_after - rss_before) / (1024*1024)
    peak_delta_mb = (peak_after - peak_before) / (1024*1024)

    print("======================================")
    print(f"Wall time: {wall:.3f} s")
    print(f"RSS Δ: {rss_delta_mb:+.2f} MB")
    print(f"Peak memory Δ: {peak_delta_mb:+.2f} MB (OS-dependent)")
    print("======================================")

    return None

In [42]:
%%timemem

from pyspark.sql import SparkSession

# Create SparkSession quietly
spark = (
    SparkSession.builder
    .appName("Assignment1")
    .master("local[*]")
    .config("spark.ui.showConsoleProgress", "false")  # disable progress bar
    .getOrCreate()
)

Wall time: 0.120 s
RSS Δ: +0.00 MB
Peak memory Δ: +0.00 MB (OS-dependent)


25/10/23 15:08:00 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


# 2. Word Count with RDDs

In [43]:
%%timemem

# TODO: Write your code below, but do not remove any lines already in this cell.

sc = spark.sparkContext
lines = sc.textFile(DATA_PATH)

# By the time we get to here, "lines" should refer to an RDD with the brand file loaded.
# Let's count the lines.

lines.count()

7262

Wall time: 0.835 s
RSS Δ: +0.00 MB
Peak memory Δ: +0.00 MB (OS-dependent)


In [44]:
%%timemem

# TODO: Write your code below, but do not remove any lines already in this cell.

# Split lines into words, lowercase, replace non-[a-z] with space, then flatten
words = lines.flatMap(lambda line: re.sub(r'[^a-z]', ' ', line.lower()).split())

words = words.filter(lambda w: len(w) >= 2)

word_pairs = words.map(lambda w: (w, 1))

word_counts_rdd = word_pairs.reduceByKey(lambda a, b: a + b)

word_counts = word_counts_rdd.collect()
word_counts.sort(key=lambda x: x[1], reverse=True)

# By the time we get to here "word_counts" already has the collected output, sorted by frequency in descending order.
# So we just print out the top-10.

for word, count in word_counts[:10]:
    print(f"{word}: {count}")

and: 16150
the: 9612
in: 7958
is: 7814
for: 6789
brand: 6476
its: 4241
to: 4026
of: 3382
with: 3099
Wall time: 0.493 s
RSS Δ: +1.00 MB
Peak memory Δ: +0.00 MB (OS-dependent)


# 3. Word Count with DataFrames

## Question : What's the difference between RDD and DataFrame

DataFrames let you work with data in a table-like format. You describe what you want to do, and Spark figures out how to do it efficiently. RDDs require you to manually specify each step of the computation. DataFrames are higher-level, easier to use, and often faster than RDDs.

In [45]:
df = spark.read.csv(
    DATA_PATH,
    header=True,
    escape='"',
    inferSchema=True
)

# By the time we get to here, the file should have already been loaded into a DataFrame.
# Here, we just inspect it.

print("Rows:", df.count())
df.printSchema()
df.select("description").show(5, truncate=80)

Rows: 7261
root
 |-- brand: string (nullable = true)
 |-- description: string (nullable = true)

+--------------------------------------------------------------------------------+
|                                                                     description|
+--------------------------------------------------------------------------------+
|a-case is a brand specializing in protective accessories for electronic devic...|
|A-Derma is a French dermatological skincare brand specializing in products fo...|
| a patented ingredient derived from oat plants cultivated under organic farmi...|
|                                                                       cleansers|
|           A-Derma emphasizes clinical efficacy and hypoallergenic formulations.|
+--------------------------------------------------------------------------------+
only showing top 5 rows


In [46]:
words_df = df.select(
    explode(
        split(
            regexp_replace(lower(col("description")), "[^a-z]", " "),
            "\s+"
        )
    ).alias("word")
)


words_df = words_df.filter(col("word").rlike(".{2,}"))


word_counts = words_df.groupBy("word").count().orderBy(col("count").desc())

# Show top 10
top10 = word_counts.limit(10)
top10.show()

+-----+-----+
| word|count|
+-----+-----+
|  and|13094|
|  the| 6895|
|   is| 6419|
|   in| 6351|
|  for| 5530|
|brand| 5196|
|  its| 3304|
|   to| 3155|
|   of| 2692|
|known| 2509|
+-----+-----+



## Question : Does the RDD approach and the DataFrame approach give the same answers? Explain why or why not.

Sometimes the last word in the top 10 is different because two or more words have the same count. Spark doesn’t always pick the same one first unless you tell it exactly how to break ties. Small differences in splitting or cleaning the text can also change which word appears last.

In [47]:
%%timemem

# TODO: Write your code below, but do not remove any lines already in this cell.

import numpy
from pyspark.ml.feature import StopWordsRemover

words_df = df.select(
    explode(
        split(
            regexp_replace(lower(col("description")), "[^a-z]", " "),
            "\s+"
        )
    ).alias("word")
).filter(col("word").rlike(".{2,}"))  # remove short words

remover = StopWordsRemover()
stopwords = remover.getStopWords()

words_filtered_df = words_df.filter(~col("word").isin(stopwords))

word_counts_noStopWords = words_filtered_df.groupBy("word").count().orderBy(col("count").desc())


# By the time we get to here "word_counts_noStopWords" is a DataFrame that already has the word counts sorted in descending order.
# So we just print out the top-10.

top10_noStopWords = word_counts_noStopWords.limit(10)
top10_noStopWords.show()

+------------+-----+
|        word|count|
+------------+-----+
|       brand| 5196|
|       known| 2509|
|    products| 2459|
|   primarily| 2100|
|      market| 1873|
|       range| 1688|
|  recognized| 1482|
|   including| 1452|
|specializing| 1390|
|       often| 1247|
+------------+-----+

Wall time: 0.513 s
RSS Δ: +0.00 MB
Peak memory Δ: +0.00 MB (OS-dependent)


In [48]:
%%timemem

# TODO: Write your code below, but do not remove any lines already in this cell.

import csv

# Collect DataFrame rows as a list of tuples
top10_list = top10.collect()
top10_noStopWords_list = top10_noStopWords.collect()

with open('/home/maxence/Documents/data_engineering/top10_words.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(top10.columns)
    for row in top10_list:
        writer.writerow(row)

with open('/home/maxence/Documents/data_engineering/top10_noStopWords.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(top10_noStopWords.columns)
    for row in top10_noStopWords_list:
        writer.writerow(row)


Wall time: 0.637 s
RSS Δ: +0.00 MB
Peak memory Δ: +0.00 MB (OS-dependent)


In [37]:
spark.stop()
print("Spark session stopped.")


Spark session stopped.
